# Partie 6 — Prime finale et soumission Kaggle

La prime pure d'un client est le produit des deux modèles :

$$\widehat{\text{Prime}}(x) = \hat p(x) \times \hat g(x)$$

où $\hat p$ est la probabilité de sinistre (notebook 01) et $\hat g$ le montant attendu si sinistre (notebook 02).

**Métrique Kaggle** (compétition CY Tech 2027, vérifiée par mon binôme sur la page de la compétition) : la **RMSE de la prime** sur le test (la consigne parle de MSE ; les deux classent les modèles dans le même ordre). Le fichier de soumission a deux colonnes : `index` et `pred`.

**Évaluation locale** : on calcule la même RMSE sur le train, avec les prédictions **hors pli** (5 plis par client, colonne `pli`). Le pli 1 correspond exactement au jeu de validation du notebook 01.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Racine du projet : le dossier qui contient Data/ (lancement depuis notebooks/ ou depuis la racine)
RACINE = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "Data" / "train.csv").exists())
PREDICTIONS = RACINE / "Data" / "predictions"

train = pd.read_csv(RACINE / "Data" / "train.csv", dtype={"code_postal": str})
test = pd.read_csv(RACINE / "Data" / "test.csv", dtype={"code_postal": str})
oof = pd.read_csv(PREDICTIONS / "binome_composantes_oof_train.csv")   # prédictions hors pli sur le train
binome_test = pd.read_csv(PREDICTIONS / "binome_composantes_test.csv")  # prédictions sur le test Kaggle
assert (oof["index"].values == train["index"].values).all() and (binome_test["index"].values == test["index"].values).all()
print("train :", train.shape, "| test :", test.shape, "| plis :", sorted(oof["pli"].unique()))

train : (50000, 33) | test : (50000, 28) | plis : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


## 1. Toutes les combinaisons fréquence × gravité disponibles

Trois fréquences : la régression logistique de référence (ma version d'avant l'audit, reprise par le binôme), ma logistique actuelle (notebook 01) et la logistique de référence corrigée par des courbes (binôme). Trois gravités. On compare la RMSE locale hors pli avec le score Kaggle public quand il existe.

In [2]:
# Ma fréquence (notebook 01) : mêmes plis, mêmes contrats que les prédictions du binôme
ma_freq_oof = pd.read_csv(PREDICTIONS / "frequence_logistique_oof_train.csv")
ma_freq_test = pd.read_csv(PREDICTIONS / "frequence_logistique_test.csv")
assert (ma_freq_oof["index"].values == oof["index"].values).all() and (ma_freq_oof["pli"].values == oof["pli"].values).all()
assert (ma_freq_test["id"].values == binome_test["index"].values).all()
oof["proba_ma_logistique"] = ma_freq_oof["proba_pred"].values
binome_test["proba_ma_logistique"] = ma_freq_test["proba_pred"].values

FREQUENCES = {"logistique de référence": "proba_logistique_reference",
              "ma logistique (notebook 01)": "proba_ma_logistique",
              "logistique + courbes": "proba_logistique_courbes"}
GRAVITES = {"Gamma de référence": "gravite_gamma_reference",
            "Gamma + XGBoost": "gravite_gamma_xgboost",
            "Gamma + courbes": "gravite_gamma_courbes"}
# Scores Kaggle publics obtenus par mon binôme (fréquence, gravité) -> score
KAGGLE = {("logistique de référence", "Gamma de référence"): 1365.943,
          ("logistique de référence", "Gamma + XGBoost"): 1365.913,
          ("logistique de référence", "Gamma + courbes"): 1365.745,
          ("logistique + courbes", "Gamma + courbes"): 1365.657}


def rmse_prime(prime, cout):
    return np.sqrt(np.mean((np.asarray(prime) - np.asarray(cout)) ** 2))


lignes = [{"fréquence": "(prime constante)", "gravité": "", "RMSE locale hors pli": rmse_prime(np.full(len(oof), oof["montant_sinistre"].mean()), oof["montant_sinistre"]), "Kaggle public": np.nan}]
for nom_f, col_f in FREQUENCES.items():
    for nom_g, col_g in GRAVITES.items():
        prime = oof[col_f] * oof[col_g]
        lignes.append({"fréquence": nom_f, "gravité": nom_g,
                       "RMSE locale hors pli": rmse_prime(prime, oof["montant_sinistre"]),
                       "Kaggle public": KAGGLE.get((nom_f, nom_g), np.nan)})
combinaisons = pd.DataFrame(lignes).sort_values("RMSE locale hors pli")
combinaisons.round(4)

,fréquence,gravité,RMSE locale hors pli,Kaggle public
9,logistique + courbes,Gamma + courbes,541.9266,1365.657
8,logistique + courbes,Gamma + XGBoost,541.9575,NaN
7,logistique + courbes,Gamma de référence,541.9659,NaN
3,logistique de référence,Gamma + courbes,541.9951,1365.745
6,ma logistique (notebook 01),Gamma + courbes,542.0162,NaN
2,logistique de référence,Gamma + XGBoost,542.0282,1365.913
1,logistique de référence,Gamma de référence,542.0441,1365.943
5,ma logistique (notebook 01),Gamma + XGBoost,542.0501,NaN
4,ma logistique (notebook 01),Gamma de référence,542.0657,NaN
0,(prime constante),,545.3974,NaN


→ **L'ordre des RMSE locales est le même que celui des scores Kaggle** pour les 4 combinaisons notées : 542,0441 → 1365.943 ; 542,0282 → 1365.913 ; 541,9951 → 1365.745 ; 541,9266 → 1365.657. Notre évaluation locale est donc un bon guide pour choisir un modèle sans avoir à soumettre.
- **Les écarts sont minuscules** : toutes les combinaisons sont entre 541,93 et 542,04, contre 545,40 pour une prime constante.
- **La correction de la fréquence pèse même plus que celle de la gravité** : avec chacune des trois gravités, passer de la logistique de référence à la logistique + courbes fait gagner de 0,068 à 0,078 en RMSE ; passer de la Gamma de référence à Gamma + courbes ne fait gagner que 0,039 à 0,049. Améliorer la fréquence est donc une piste utile.
- **Ma logistique actuelle (notebook 01) fait un peu moins bien que la logistique de référence** sur la prime : 542,0162 contre 541,9951 avec Gamma + courbes, soit +0,021 (et +0,022 avec les deux autres gravités). Leur log-loss est pourtant presque identique (0,21402 contre 0,21399, notebook 01, 4.7). L'écart reste à tester pli par pli avant de conclure.

## 2. Cohérence globale : l'assureur serait-il rentable ?

Pour la meilleure combinaison, on compare sur le train (hors pli) la **somme des primes prédites** à la **somme des sinistres observés**, au total et pli par pli. Un ratio proche de 1 signifie que les primes couvrent en moyenne le coût des sinistres.

In [3]:
FREQ_RETENUE, GRAV_RETENUE = "proba_logistique_courbes", "gravite_gamma_courbes"

oof["prime"] = oof[FREQ_RETENUE] * oof[GRAV_RETENUE]
coherence = oof.groupby("pli").agg(primes_predites=("prime", "sum"), sinistres_observes=("montant_sinistre", "sum"))
coherence.loc["total"] = coherence.sum()
coherence["ratio primes / sinistres"] = coherence["primes_predites"] / coherence["sinistres_observes"]
print(f"Prime moyenne prédite : {oof['prime'].mean():.2f} € | coût moyen observé : {oof['montant_sinistre'].mean():.2f} €")
coherence.round(3)

Prime moyenne prédite : 103.35 € | coût moyen observé : 103.37 €


,primes_predites,sinistres_observes,ratio primes / sinistres
pli,,,
1,1029215.154,1039292.53,0.990
2,1023526.867,1054345.65,0.971
3,1031746.725,1003627.18,1.028
4,1046515.213,998930.30,1.048
5,1036402.479,1072271.54,0.967
total,5167406.438,5168467.20,1.000


→ Sur l'ensemble du train, **les primes prédites couvrent les sinistres observés** : ratio de 1,000, prime moyenne de 103,35 € pour un coût moyen de 103,37 €. L'assureur serait rentable en moyenne, avant frais et marge.

Pli par pli, le ratio varie de 0,967 à 1,048 : sur 10 000 contrats, quelques gros sinistres suffisent à faire varier le coût total d'environ ± 5 %.

## 3. Soumission Kaggle

Les modèles retenus ont été **réentraînés sur les 50 000 contrats** du train avant de prédire le test (fichier `binome_composantes_test.csv`). On écrit `submission.csv` à la racine du projet, au format attendu : `index`, `pred`.

Pour l'instant, la combinaison retenue est la meilleure connue (logistique + courbes × Gamma + courbes, 1365.657 sur Kaggle). Quand le notebook 01 produira une nouvelle fréquence, il suffira de l'ajouter au tableau de la section 1 (mêmes 5 plis, colonne `pli`) et de la choisir ici si elle fait mieux.

In [4]:
submission = pd.DataFrame({"index": binome_test["index"],
                           "pred": binome_test[FREQ_RETENUE] * binome_test[GRAV_RETENUE]})

# Contrôles du format attendu par Kaggle
assert len(submission) == 50000 and submission["index"].is_unique
assert submission["index"].min() == 50000 and submission["index"].max() == 99999
assert np.isfinite(submission["pred"]).all() and (submission["pred"] >= 0).all()

submission.to_csv(RACINE / "submission.csv", index=False)
print(f"submission.csv écrit : {len(submission)} lignes, prime moyenne {submission['pred'].mean():.2f} €")
submission.head()

submission.csv écrit : 50000 lignes, prime moyenne 102.71 €


,index,pred
0,50000,54.144409
1,50001,40.833700
2,50002,215.189921
3,50003,119.202512
4,50004,33.450054
